In [1]:
import polars as pl
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
import seaborn as sns

from src.db import PBWarehouse
from sklearn.metrics import classification_report, confusion_matrix

2025-10-02 14:34:02.986 | INFO     | src.config:<module>:28 - Loaded environment variables from /home/iragca/Documents/github/capstone-project-2/.env
2025-10-02 14:34:02.986 | INFO     | src.config:<module>:67 - PROJECT_ROOT: /home/iragca/Documents/github/capstone-project-2
2025-10-02 14:34:02.987 | INFO     | src.config:<module>:68 - DATA_DIR: /home/iragca/Documents/github/capstone-project-2/data


# Loading the dataset

In [2]:
pb = PBWarehouse()
dataset = pb.get_dataset("annotated_tweets")


# is_extremist is boolean where True means extremist
# We want to convert it to is_hateful where 0 means extremist
# and 1 means not extremist
# We want to map the booleans to the correct HateBERT class labels
correct_labels = {
    0: 1,
    1: 0,
    # boolean: hatebert label
}

df = pl.DataFrame(
    {
        "text": [record.text for record in dataset],
        "is_extremist": [
            correct_labels[int(record.is_extremist)] for record in dataset
        ],
    }
)
df

text,is_extremist
str,i64
"""""All the cops are gone so it’s…",1
"""""...The disproportionate death…",1
"""""Although my school addressed …",1
"""""#Antiracism is the active pro…",1
"""""...The desire to control trut…",1
…,…
"""A story from @washingtonpost …",1
"""@Steph_I_Will I hope they gave…",1
"""The Resistance &amp; I release…",1


In [10]:
df["is_extremist"].value_counts()

is_extremist,count
i64,u32
1,1694
0,6


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"].to_list(),
    df["is_extremist"].to_list(),
    test_size=0.2,
    random_state=42,
    stratify=df["is_extremist"].to_list()
)

# Finding the base performance

Let's use the test set to get the initial performance before finetuning.

In [20]:
MODEL_NAME = "Hate-speech-CNERG/bert-base-uncased-hatexplain"
# HATEBERT on Hugging Face

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, attn_implementation="eager"
)

In [21]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def predict(text: str) -> int:
    inputs = tokenizer(text, return_tensors="pt", padding=True)
    inputs.to(DEVICE)
    model.to(DEVICE)
    outputs = model(**inputs)
    logits = outputs.logits
    label = torch.argmax(torch.nn.functional.softmax(logits, dim=-1), dim=-1)
    return label.item()

In [22]:
initial_test_preds = [predict(text) for text in X_test]

print(classification_report(y_test, initial_test_preds, digits=4))

              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000         1
           1     1.0000    0.9735    0.9865       339
           2     0.0000    0.0000    0.0000         0

    accuracy                         0.9706       340
   macro avg     0.3333    0.3245    0.3288       340
weighted avg     0.9971    0.9706    0.9836       340



/home/iragca/Documents/github/capstone-project-2/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/iragca/Documents/github/capstone-project-2/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/iragca/Documents/github/capstone-project-2/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _war

In [ ]:
import numpy as np

# find all classes that are equal to '0' (extremist)
abs(np.array(y_train) - 1).sum()

np.int64(5)

Initial performance:

- missed all extreimist classes (0 f1 score for class '0')
- near perfect performance on class '1' (normal / non-extremist)

# Finetuning

In [7]:
# Freeze all layers except the classification head
for param in model.bert.parameters():
    param.requires_grad = False


for param in model.classifier.parameters():
    param.requires_grad = True

In [8]:
EPOCHS = 3
BATCH_SIZE = 16
LR = 5e-5

# 1. Tokenize everything at once
encodings = tokenizer(
    X_train,
    return_tensors="pt"
    , padding=True,
)

labels = torch.tensor(y_train)

dataset = TensorDataset(
    encodings["input_ids"],
    encodings["attention_mask"],
    labels
)

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = torch.nn.CrossEntropyLoss()

total_steps = len(loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

def train(model, loader, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for step, batch in enumerate(loader):
            input_ids, attention_mask, labels = [x.to(device) for x in batch]

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss
            total_loss += loss.item()

            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            if step % 10 == 0:
                print(f"Epoch {epoch+1}/{epochs}, Step {step}, Loss: {loss.item():.4f}")

        avg_loss = total_loss / len(loader)
        print(f"Epoch {epoch+1} finished. Avg Loss: {avg_loss:.4f}")

train(model, loader, EPOCHS)


NameError: name 'TensorDataset' is not defined